## Langchain Types

In [ ]:
!pip install -q langchain langchain-groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.8 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("Groq_API")

## Simple Chain

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os


model = ChatGroq(
    model="llama-3.3-70b-versatile",   # or another Groq-supported model
    groq_api_key=os.getenv("GROQ_API_KEY")
)

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a facts expert who knows facts about {animal}."),
        ("human", "Tell me {fact_count} facts."),
    ]
)

chain = prompt_template | model | StrOutputParser()


result = chain.invoke({
    "animal": "elephant",
    "fact_count": 3
})

print(result)

Here are three facts about elephants:

1. **Elephant Memory**: Elephants have an excellent memory. They are known to have a strong recall of their family members, even after many years of separation. In fact, they have been observed showing signs of grief and recognition when reunited with their family members after a long time.

2. **Trunk Functionality**: An elephant's trunk is a highly versatile and important part of their body. It is used for breathing, drinking, eating, grasping objects, and even social interactions like greeting and showing affection. The trunk contains many muscles, giving it a high degree of flexibility and dexterity.

3. **Social Structure**: Elephants are highly social animals that live in large matriarchal herds, led by the oldest female. These herds are typically made up of related females and their offspring, while male elephants often leave their natal herd as they mature and may live solitary lives or form bachelor groups. This social structure is crucia

In [ ]:
import gradio as gr

def get_facts(animal, count):
    return chain.invoke({
        "animal": animal,
        "fact_count": count
    })

demo = gr.Interface(
    fn=get_facts,
    inputs=[
        gr.Textbox(label="Animal"),
        gr.Slider(1, 10, value=3, step=1, label="Number of Facts")
    ],
    outputs="text",
    title="Animal Facts with Groq + LangChain"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://22695e19ac3e7b0caa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Sequential Chain / Extendable chain

In [ ]:
import gradio as gr
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
import os

# LLM
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=os.getenv("GROQ_API_KEY")
)

# Prompt for facts
animal_facts_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You like telling facts and you tell facts about {animal}."),
        ("human", "Tell me {count} facts."),
    ]
)

# Prompt for translation
translation_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a translator and convert the provided text into {language}."),
        ("human", "Translate the following text to {language}: {text}"),
    ]
)

# Runnable
prepare_for_translation = RunnableLambda(
    lambda output: {
        "text": output,
        "language": "French"
    }
)

# Chain
chain = (
    animal_facts_template
    | model
    | StrOutputParser()
    | prepare_for_translation
    | translation_template
    | model
    | StrOutputParser()
)

# Gradio function
def translate_facts(animal, count):
    return chain.invoke({
        "animal": animal,
        "count": int(count)
    })

# Gradio UI
demo = gr.Interface(
    fn=translate_facts,
    inputs=[
        gr.Textbox(label="Animal", placeholder="e.g. Cat"),
        gr.Number(label="Number of Facts", value=2)
    ],
    outputs=gr.Textbox(label="Translated Facts"),
    title="Animal Facts Translator",
    description="Generate facts about an animal and translate them into French."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b6b630b02af8afc7ec.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def translate(text, language):
    return chain.invoke({
        "text": text,
        "language": language
    })
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌍 AI Language Translator")

    text = gr.Textbox(
        label="Enter Text",
        lines=6,
        placeholder="Type something..."
    )

    language = gr.Dropdown(
        ["French", "Spanish", "German", "Urdu", "Arabic", "Hindi"],
        value="French",
        label="Translate To"
    )

    translate_btn = gr.Button("Translate")

    output = gr.Textbox(
        label="Translated Text",
        lines=6
    )

    translate_btn.click(
        fn=translate,
        inputs=[text, language],
        outputs=output
    )

demo.launch()

/tmp/ipykernel_989/2588130348.py:8: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7ded7da518475f2f45.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Conditional Chains

In [ ]:
from langchain_core.runnables import RunnableBranch


# Positive Feedback Prompt
positive_feedback_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Generate a thank you note for this positive feedback: {feedback}.")
])

# Negative Feedback Prompt
negative_feedback_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Generate a response addressing this negative feedback: {feedback}.")
])

# Neutral Feedback Prompt
neutral_feedback_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Generate a request for more details for this neutral feedback: {feedback}.")
])

# Escalation Prompt
escalate_feedback_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "Generate a message to escalate this feedback to a human agent: {feedback}.")
])

# Classification Prompt
classification_template = ChatPromptTemplate.from_messages([
    ("system", "You are a sentiment classifier."),
    ("human", "Classify this feedback as one word only: positive, negative, neutral, or escalate.\n\nFeedback: {feedback}")
])

# Branches
branches = RunnableBranch(
    (
        lambda x: "positive" in x.lower(),
        positive_feedback_template | model | StrOutputParser()
    ),
    (
        lambda x: "negative" in x.lower(),
        negative_feedback_template | model | StrOutputParser()
    ),
    (
        lambda x: "neutral" in x.lower(),
        neutral_feedback_template | model | StrOutputParser()
    ),
    escalate_feedback_template | model | StrOutputParser()
)

# Classification Chain
classification_chain = classification_template | model | StrOutputParser()

# Complete Chain
chain = classification_chain | branches

# Example Review
review = "The product is terrible. It broke after just one use and the quality is very poor."

# Invoke
result = chain.invoke({
    "feedback": review
})

print(result)

I'm so sorry to hear that you've had a negative experience. Can you please provide more details about what went wrong? I'm here to listen and help in any way I can. Your feedback is valuable to me, and I'll do my best to understand what happened and make things right. What specifically didn't meet your expectations, and how can I improve moving forward?


In [ ]:
# ------------------- Gradio Function -------------------
def analyze_feedback(feedback):
    if not feedback.strip():
        return "Please enter feedback."

    return chain.invoke({
        "feedback": feedback
    })
# ------------------- Gradio UI -------------------
demo = gr.Interface(
    fn=analyze_feedback,
    inputs=gr.Textbox(
        lines=6,
        label="Customer Feedback",
        placeholder="Enter customer feedback here..."
    ),
    outputs=gr.Textbox(
        lines=8,
        label="AI Response"
    ),
    title="📝 AI Feedback Response Generator",
    description="Classifies customer feedback and generates an appropriate response using RunnableBranch."
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://44764634e0c843c036.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://22695e19ac3e7b0caa.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://b6b630b02af8afc7ec.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://44764634e0c843c036.gradio.live


## Parallel Chaining

In [ ]:
import os
import gradio as gr

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

In [ ]:
# Summary Prompt
summary_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a movie critic."),
        ("human", "Provide a brief summary of the movie {movie_name}."),
    ]
)

# Plot Analysis Prompt
def analyze_plot(plot):
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a movie critic."),
            ("human","Analyze the plot:\n\n{plot}\n\nWhat are its strengths and weaknesses?",
             ),
        ]
    )
    return prompt.format_prompt(plot=plot)

# Character Analysis Prompt
def analyze_characters(characters):
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a movie critic."),
            ("human","Analyze the characters:\n\n{characters}\n\nWhat are their strengths and weaknesses?",
            ),
        ]
    )
    return prompt.format_prompt(characters=characters)

# Combine Results
def combine_verdicts(plot_analysis, character_analysis):
    return f"""
## Plot Analysis

{plot_analysis}

---

## Character Analysis

{character_analysis}
"""

# Branches
plot_branch_chain = (
    RunnableLambda(analyze_plot)
    | model
    | StrOutputParser()
)

character_branch_chain = (
    RunnableLambda(analyze_characters)
    | model
    | StrOutputParser()
)

# Main Chain
chain = (
    summary_template
    | model
    | StrOutputParser()
    | RunnableParallel(
        branches={
            "plot": plot_branch_chain,
            "characters": character_branch_chain,
        }
    )
    | RunnableLambda(
        lambda x: combine_verdicts(
            x["branches"]["plot"],
            x["branches"]["characters"],
        )
    )
)

In [ ]:
def movie_review(movie_name):
    try:
        return chain.invoke({"movie_name": movie_name})
    except Exception as e:
        return str(e)
demo = gr.Interface(
    fn=movie_review,
    inputs=gr.Textbox(
        label="Movie Name",
        placeholder="Enter a movie name..."
    ),
    outputs=gr.Markdown(label="Movie Analysis"),
    title="🎬 Movie Critic using LangChain",
    description="Enter a movie name to generate a summary, plot analysis, and character analysis."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://926f2011c80365d62f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
